In [ ]:
import pandas as pd
import numpy as np
import torch

from pathlib import Path
from datasets import Dataset as HFDataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)

from sentence_transformers.training_args import BatchSamplers

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cpu
CUDA available: False


C:\Users\user\AppData\Local\Temp\ipykernel_25736\3919144876.py:8: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
C:\Users\user\AppData\Local\Temp\ipykernel_25736\3919144876.py:15: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


In [ ]:
def select_group_safe_pairs(fit_pairs):
    pairs = fit_pairs.copy()

    parent = {}

    def find(x):
        parent.setdefault(x, x)

        if parent[x] != x:
            parent[x] = find(parent[x])

        return parent[x]

    def union(a, b):
        ra = find(a)
        rb = find(b)

        if ra != rb:
            parent[rb] = ra

    for source, target in pairs[
        ["source_id", "target_id"]
    ].itertuples(index=False, name=None):

        union(str(source), str(target))

    pairs["train_group"] = [
        find(str(source))
        for source in pairs["source_id"]
    ]

    selected = (
        pairs
        .sort_values(["source_id", "target_id"])
        .drop_duplicates(subset=["train_group"])
        .reset_index(drop=True)
    )

    return selected

In [5]:
fit_pairs = pd.read_csv(
    processed_dir / "issuedupes_fit_pairs.csv",
    dtype={"source_id": str, "target_id": str}
)

safe_pairs = select_group_safe_pairs(fit_pairs)

print("Original training pairs:", len(fit_pairs))
print("Selected training pairs:", len(safe_pairs))
print("Unique training groups:", safe_pairs["train_group"].nunique())

assert safe_pairs["train_group"].is_unique

train_dataset = HFDataset.from_dict({
    "anchor": safe_pairs["text_a"].astype(str).tolist(),
    "positive": safe_pairs["text_b"].astype(str).tolist()
})

print("Training examples:", len(train_dataset))
print(train_dataset.column_names)

Original training pairs: 6439
Selected training pairs: 5936
Unique training groups: 5936
Training examples: 5936
['anchor', 'positive']


In [6]:
model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

model.max_seq_length = 256

loss = losses.MultipleNegativesRankingLoss(model)

print("Model:", MODEL_NAME)
print("Device:", model.device)
print("Maximum sequence length:", model.max_seq_length)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Maximum sequence length: 256


In [7]:
training_args = SentenceTransformerTrainingArguments(
    output_dir=str(
        project_root / "models" / "issuedupes_minilm_pilot"
    ),

    max_steps=50,
    per_device_train_batch_size=8,

    learning_rate=2e-5,
    warmup_steps=5,

    fp16=False,
    bf16=False,
    use_cpu=True,

    batch_sampler=BatchSamplers.NO_DUPLICATES,

    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",

    report_to="none",
    seed=42,
    data_seed=42
)

In [8]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss
)

trainer.train()

pilot_path = (
    project_root
    / "models"
    / "issuedupes_minilm_pilot"
    / "final"
)

model.save(str(pilot_path))

print("Pilot model saved to:", pilot_path)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
10,0.044006
20,0.119778
30,0.021971
40,0.084108
50,0.048605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pilot model saved to: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\models\issuedupes_minilm_pilot\final


In [ ]:
issues = pd.read_csv(
    processed_dir / "issuedupes_issues_grouped.csv",
    dtype={"issue_id": str}
)

links = pd.read_csv(
    processed_dir / "issuedupes_direct_links.csv",
    dtype={"source_id": str, "target_id": str}
)

issues["created_at"] = pd.to_datetime(
    issues["created_at"], utc=True
)

issues["text"] = issues["text"].fillna("")

issues = issues[
    issues["text"].str.strip().ne("")
    & issues["created_at"].notna()
].sort_values(
    ["created_at", "issue_id"]
).reset_index(drop=True)

outer_split = int(len(issues) * 0.8)
outer_cutoff = issues.iloc[outer_split]["created_at"]

training_period = issues[
    issues["created_at"] < outer_cutoff
].reset_index(drop=True)

inner_split = int(len(training_period) * 0.85)

inner_cutoff = training_period.iloc[
    inner_split
]["created_at"]

validation = training_period[
    training_period["created_at"] >= inner_cutoff
].copy()

valid_ids = set(training_period["issue_id"])

adjacency = {}

for source, target in links[
    ["source_id", "target_id"]
].itertuples(index=False, name=None):

    if source == target:
        continue

    if source not in valid_ids or target not in valid_ids:
        continue

    adjacency.setdefault(source, set()).add(target)
    adjacency.setdefault(target, set()).add(source)

created_lookup = training_period.set_index(
    "issue_id"
)["created_at"].to_dict()

relevant = {}

for row in validation.itertuples(index=False):

    earlier = {
        other
        for other in adjacency.get(row.issue_id, set())
        if created_lookup[other] < row.created_at
    }

    if earlier:
        relevant[row.issue_id] = earlier

eligible = validation[
    validation["issue_id"].isin(relevant)
].copy()

print("Training-period issues:", len(training_period))
print("Validation issues:", len(validation))
print("Eligible validation queries:", len(eligible))

Training-period issues: 15275
Validation issues: 2292
Eligible validation queries: 1236


In [10]:
from sentence_transformers import SentenceTransformer

fine_tuned_path = (
    project_root
    / "models"
    / "issuedupes_minilm_pilot"
    / "final"
)

fine_tuned_model = SentenceTransformer(
    str(fine_tuned_path),
    device="cpu"
)

fine_tuned_model.max_seq_length = 256

print("Fine-tuned model loaded:", fine_tuned_path)

fine_tuned_embeddings = fine_tuned_model.encode(
    training_period["text"].tolist(),
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", fine_tuned_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned model loaded: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\models\issuedupes_minilm_pilot\final


Batches:   0%|          | 0/955 [00:00<?, ?it/s]

Embedding shape: (15275, 384)


In [11]:
def evaluate_validation(issues, eligible, relevant, embeddings):

    issue_ids = issues["issue_id"].to_numpy()

    timestamps = issues["created_at"].to_numpy(
        dtype="datetime64[ns]"
    )

    id_to_idx = {
        issue_id: idx
        for idx, issue_id in enumerate(issue_ids)
    }

    ranks = []

    for query in eligible.itertuples(index=False):

        query_idx = id_to_idx[query.issue_id]

        cutoff = np.searchsorted(
            timestamps,
            np.datetime64(query.created_at.to_datetime64()),
            side="left"
        )

        scores = embeddings[:cutoff] @ embeddings[query_idx]

        relevant_indices = [
            id_to_idx[rid]
            for rid in relevant[query.issue_id]
            if rid in id_to_idx
            and id_to_idx[rid] < cutoff
        ]

        if not relevant_indices:
            raise ValueError("No eligible relevant report")

        best_score = max(scores[idx] for idx in relevant_indices)

        best_idx = min(
            idx for idx in relevant_indices
            if scores[idx] == best_score
        )

        rank = (
            np.count_nonzero(scores > best_score)
            + np.count_nonzero(scores[:best_idx] == best_score)
            + 1
        )

        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "Queries": len(ranks),
        "Hit@1": np.mean(ranks <= 1),
        "Hit@5": np.mean(ranks <= 5),
        "Hit@10": np.mean(ranks <= 10),
        "MRR": np.mean(1.0 / ranks)
    }

In [12]:
fine_tuned_results = evaluate_validation(
    issues=training_period,
    eligible=eligible,
    relevant=relevant,
    embeddings=fine_tuned_embeddings
)

print("Fine-tuned Sentence-BERT validation results:")

display(
    pd.DataFrame(
        [fine_tuned_results],
        index=["Fine-tuned MiniLM"]
    ).round(4)
)

Fine-tuned Sentence-BERT validation results:


,Queries,Hit@1,Hit@5,Hit@10,MRR
Fine-tuned MiniLM,1236,0.6311,0.8301,0.8706,0.7189


In [13]:
pretrained_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

pretrained_model.max_seq_length = 256

print("Original pretrained model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Original pretrained model loaded


In [ ]:
embeddings_dir = project_root / "models" / "embeddings"

pretrained_embeddings_all = np.load(
    embeddings_dir / "issuedupeslite_embeddings.npy",
    mmap_mode="r"
)

saved_ids = pd.read_csv(
    embeddings_dir / "issuedupeslite_issue_ids.csv",
    dtype={"issue_id": str}
)["issue_id"].tolist()

id_to_embedding_idx = {
    issue_id: idx
    for idx, issue_id in enumerate(saved_ids)
}

assert set(training_period["issue_id"]).issubset(
    id_to_embedding_idx
), "Some training-period issues are missing from saved embeddings"

pretrained_embeddings = np.asarray(
    pretrained_embeddings_all[
        [
            id_to_embedding_idx[issue_id]
            for issue_id in training_period["issue_id"]
        ]
    ]
)

print("Pretrained embedding shape:", pretrained_embeddings.shape)

assert pretrained_embeddings.shape == fine_tuned_embeddings.shape

Pretrained embedding shape: (15275, 384)


In [15]:
pretrained_results = evaluate_validation(
    issues=training_period,
    eligible=eligible,
    relevant=relevant,
    embeddings=pretrained_embeddings
)

print("Pretrained MiniLM validation results:")

display(
    pd.DataFrame(
        [pretrained_results],
        index=["Pretrained MiniLM"]
    ).round(4)
)

Pretrained MiniLM validation results:


,Queries,Hit@1,Hit@5,Hit@10,MRR
Pretrained MiniLM,1236,0.6311,0.8285,0.8689,0.7159


In [16]:
comparison_df = pd.DataFrame({
    "Pretrained MiniLM": pretrained_results,
    "Fine-tuned MiniLM": fine_tuned_results
}).T

comparison_df["Hit@1"] *= 100
comparison_df["Hit@5"] *= 100
comparison_df["Hit@10"] *= 100

display(comparison_df.round(4))

,Queries,Hit@1,Hit@5,Hit@10,MRR
Pretrained MiniLM,1236.0,63.1068,82.8479,86.8932,0.7159
Fine-tuned MiniLM,1236.0,63.1068,83.0097,87.0550,0.7189


In [17]:
metrics = ["Hit@1", "Hit@5", "Hit@10", "MRR"]

improvement = (
    comparison_df.loc["Fine-tuned MiniLM", metrics]
    - comparison_df.loc["Pretrained MiniLM", metrics]
)

print("Fine-tuning improvement:")

display(
    improvement.to_frame("Difference").round(4)
)

Fine-tuning improvement:


,Difference
Hit@1,0.0000
Hit@5,0.1618
Hit@10,0.1618
MRR,0.0030
